# Trening recognizera Kraken v2 — IMPACT-Polish v2

Cel: recognizer fine-tunowany na zbiorach polskich linii OCR, ewaluowany **wylacznie na zamroznonym
tescie IMPACT-Polish v2** (36 stron, CER/Tesseract baseline: 33.11% / 81.63%).

Dane treningowe (kolekcje train/val benchmarku, zero stron testowych):
- `PiotrSty/ocr-pl-lines` (linie polskiego OCR, split wewnetrzny)
- `PiotrSty/ehri-pl-lines` (pismo maszynowe EHRI, split per dokument)
- `PiotrSty/impact-psnc-polish-ocr` regions (cropy regionow v1, split per kolekcja)

Format treningu: `ketos train -f path` — obraz + sasiedni `.gt.txt`, listy sciezek przez `-t`/`-e`.
Kolejnosc komorek: 1 -> 7. Wymagane GPU (T4 wystarczy). Na koniec wynik i model leca na HF.

In [ ]:
# 1. Srodowisko: kraken 7.1.1 + pillow 11.3 (pin z kaggle_kraken_segtrain)
import subprocess, sys, os
from pathlib import Path
for _w in ('/kaggle/working', '/teamspace/studios/this_studio', '/content', '/workspace'):
    if Path(_w).exists():
        WORKDIR = Path(_w)
        break
else:
    WORKDIR = Path.cwd()
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'transformers'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'kraken==7.1.1', 'jiwer',
                'huggingface_hub'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', 'pillow==11.3.0'], check=True)
for _mod in list(sys.modules):
    if _mod == 'huggingface_hub' or _mod.startswith('huggingface_hub.') \
       or _mod == 'PIL' or _mod.startswith('PIL.'):
        del sys.modules[_mod]
from importlib.metadata import version as _v
print('WORKDIR:', WORKDIR)
print('IMPORT_OK kraken', _v('kraken'), '| pillow', _v('pillow'))
import torch
assert torch.cuda.is_available(), 'GPU required'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Pobranie datasetow treningowych z HF
import subprocess, sys
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

dl = WORKDIR / 'dl'
dl.mkdir(exist_ok=True)

def untar(fname, dest):
    import tarfile
    path = hf_hub_download('PiotrSty/ocr-pl-lines' if 'ocr-pl' in fname else 'PiotrSty/ehri-pl-lines',
                           fname, repo_type='dataset')
    with tarfile.open(path) as tar:
        tar.extractall(dest)
    print(fname, '->', dest)

untar('ocr-pl-lines-v1.tar.gz', WORKDIR / 'ocr_pl_lines')
untar('ehri-pl-lines-v1.tar.gz', WORKDIR / 'ehri_pl_lines')
snapshot_download('PiotrSty/impact-psnc-polish-ocr', repo_type='dataset',
                  local_dir=dl / 'impact_regions',
                  allow_patterns=['regions/train/images/*', 'regions/train/metadata.jsonl',
                                  'regions/validation/images/*', 'regions/validation/metadata.jsonl'])
print('Datasety pobrane')

In [ ]:
# 3. Budowa GT z filtrami jakosci: gt/<split>/<uuid>.png + .gt.txt (NFC)
# Lekcja z proby #1: v1-regions zawiera ornauty/inicialy z tekstem GT (0 parowania),
# ocr-pl-lines dominuje ilosciowo -> filtr tekst + filtr obrazu + cap proporcji.
import json, hashlib, random, unicodedata
from pathlib import Path
import shutil

random.seed(42)
gt = WORKDIR / 'gt'
for d in ('train', 'val'):
    (gt / d).mkdir(parents=True, exist_ok=True)
seen = set()
counters = {'train': 0, 'val': 0}
REJECTED = {'garbage_text': 0, 'image_shape': 0, 'too_long': 0, 'dup': 0}

import re
_POLISH_CHARS = set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')
_VOWELS = set('aeiouy\u0105\u0119\u00f3AEIOUY\u0104\u0118\u00d3')

def gt_ok(text):
    text = unicodedata.normalize('NFC', text).strip()
    if not text or len(text) > 300:
        return None
    tokens = [t for t in re.findall(r'\S+', text) if len(t) >= 4]
    if tokens:
        vowelless = sum(1 for t in tokens if not (set(t) & _VOWELS)) / len(tokens)
        if vowelless > 0.3:
            return None
    # udzial znakow spoza pl/lat/cyfr/interpunkcji: odrzuca cyrillice, kana, ornamenty
    ok_chars = sum(c.isascii() or c in _POLISH_CHARS or c.isspace() or c in ',.:;!?()\u201e\u201d\"\'-/' for c in text)
    if ok_chars / max(len(text), 1) < 0.92:
        return None
    return text

def add_pair(img_bytes, text, split, source=''):
    text = gt_ok(text)
    if text is None:
        REJECTED['garbage_text'] += 1
        return False
    key = hashlib.sha256(img_bytes + text.encode()).hexdigest()
    if key in seen:
        REJECTED['dup'] += 1
        return False
    seen.add(key)
    name = f'{split}_{counters[split]:05d}'
    (gt / split / f'{name}.png').write_bytes(img_bytes)
    (gt / split / f'{name}.gt.txt').write_text(text, encoding='utf-8')
    counters[split] += 1
    return True

def img_ok(img_path):
    from PIL import Image
    try:
        with Image.open(img_path) as im:
            w, h = im.size
        if h < 20 or w / max(h, 1) > 40:
            return False
        return True
    except Exception:
        return False

# ocr-pl-lines: train/ val/ z para .png + .txt (CAP 1500 na split)
ocr_dir = WORKDIR / 'ocr_pl_lines'
if ocr_dir.exists():
    for split, dest in (('train', 'train'), ('val', 'val')):
        pairs = [img for img in sorted((ocr_dir / split).glob('*.png')) if (img.with_suffix('.txt')).exists()]
        random.shuffle(pairs)
        added = 0
        for img in pairs:
            if dest == 'train' and added >= 1500:
                REJECTED['image_shape'] += len(pairs) - added
                break
            if not img_ok(img):
                REJECTED['image_shape'] += 1
                continue
            if add_pair(img.read_bytes(), img.with_suffix('.txt').read_text(encoding='utf-8'), dest):
                added += 1

# ehri-pl-lines: manifest.jsonl (image, text_file, split)
ehri = WORKDIR / 'ehri_pl_lines'
mfile = ehri / 'manifest.jsonl'
if not mfile.exists():
    import huggingface_hub
    pth = huggingface_hub.hf_hub_download('PiotrSty/ehri-pl-lines', 'manifest.jsonl', repo_type='dataset')
    shutil.copy(pth, mfile)
for line in mfile.read_text(encoding='utf-8').splitlines():
    row = json.loads(line)
    split = {'train': 'train', 'dev': 'val', 'test': 'val'}[row['split']]
    img = ehri / row['image']
    txt = ehri / row['text_file']
    if img.exists() and txt.exists() and img_ok(img):
        add_pair(img.read_bytes(), txt.read_text(encoding='utf-8'), split)

# v1 regions: bbox + tekst MUSZA istniec; ornamenty wypadaja na filtrze obrazu
import PIL.Image
for split in ('train', 'validation'):
    meta = dl / 'impact_regions' / 'regions' / split / 'metadata.jsonl'
    if not meta.exists():
        continue
    for line in meta.read_text(encoding='utf-8').splitlines():
        row = json.loads(line)
        img = meta.parent / row['file_name']
        if img.exists() and img_ok(img):
            add_pair(img.read_bytes(), row['text'], 'train' if split == 'train' else 'val')

print('Pary GT:', counters)
print('Odrzucone:', REJECTED)
for split in ('train', 'val'):
    imgs = sorted((gt / split).glob('*.png'))
    assert imgs, f'Brak GT w {split}'
    mf = WORKDIR / f'{split}_manifest.txt'
    mf.write_text('\n'.join(str(p.resolve()) for p in imgs), encoding='utf-8')
    print(split, len(imgs), '->', mf)

In [ ]:
# 4. Bazowy recognizer (ten sam start co w segtrain notebooku)
from huggingface_hub import hf_hub_download
kraken_model_path = hf_hub_download('PiotrSty/ehri-dataset', 'models/polish_nfd_9313.mlmodel',
                                    repo_type='dataset')
print('Base model:', kraken_model_path)

In [ ]:
# 5. Trening: ketos train -f path, early stopping, augmentacja
import subprocess, shutil, glob, re
output_model = str(WORKDIR / 'kraken_rec_v2')
ketos_bin = shutil.which('ketos') or 'ketos'
cmd = [
    ketos_bin, '-d', 'cuda:0', '--workers', '2',
    'train',
    '-f', 'path',
    '-t', str(WORKDIR / 'train_manifest.txt'),
    '-e', str(WORKDIR / 'val_manifest.txt'),
    '-i', kraken_model_path,
    '--resize', 'union',
    '-u', 'NFC',
    '--augment',
    '-B', '4',
    '-q', 'early', '--lag', '5',
    '-N', '100',
    '-o', output_model,
    '--weights-format', 'safetensors',
]
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print('STDOUT tail:', result.stdout[-3000:])
print('STDERR tail:', result.stderr[-3000:])
print('Return code:', result.returncode)
assert result.returncode == 0, 'ketos train niepowodzenie - zob. STDERR'
_scored = [(float(m.group(1)), p) for p in glob.glob(output_model + '/*.safetensors')
           if (m := re.search(r'(\d+\.\d+)\.safetensors$', p))]
assert _scored, 'Brak wytrenowanych wag'
best_model = max(_scored)[1]
print('Best model:', best_model)

In [ ]:
# 6. Eval e2e na zamroznonym tescie IMPACT-Polish v2 (default segmentacja + nowy recognizer)
import tarfile, json, hashlib
from huggingface_hub import hf_hub_download
BUNDLE = hf_hub_download('PiotrSty/impact-print-v2', 'impact-print-v2-test.tar.gz', repo_type='dataset')
bench = WORKDIR / 'impact-print-v2'
if not (bench / 'test_manifest.jsonl').exists():
    with tarfile.open(BUNDLE) as tar:
        tar.extractall(WORKDIR)
mpath = bench / 'test_manifest.jsonl'
records = [json.loads(l) for l in mpath.read_text(encoding='utf-8').splitlines() if l.strip()]
def _img(r):
    rel = r['image'].replace('\\', '/').split('impact-corpus/')[-1]
    for base in (WORKDIR, bench):
        p = base / 'impact-corpus' / rel
        if p.exists(): return p
        p = base / rel
        if p.exists(): return p
    raise FileNotFoundError(rel)
for r in records:
    p = _img(r)
    assert hashlib.sha256(p.read_bytes()).hexdigest() == r['sha256']
print('Test: OK,', len(records), 'stron')

import unicodedata, warnings
from PIL import Image
from kraken.tasks import SegmentationTaskModel, RecognitionTaskModel
from kraken.configs import SegmentationInferenceConfig, RecognitionInferenceConfig
warnings.filterwarnings('ignore')
seg = SegmentationTaskModel.load_model()  # wbudowany blla
rec = RecognitionTaskModel.load_model(best_model)
refs, hyps = [], []
for r in records:
    img = Image.open(_img(r)).convert('L')
    segmentation = seg.predict(img, SegmentationInferenceConfig(accelerator='cuda', device=[0]))
    pred = rec.predict(img, segmentation, RecognitionInferenceConfig(accelerator='cuda', device=[0]))
    hyp = '\n'.join(rec_.prediction.strip() for rec_ in pred)
    refs.append(unicodedata.normalize('NFC', r['text']))
    hyps.append(unicodedata.normalize('NFC', hyp))
import jiwer
cer = jiwer.cer(refs, hyps); wer = jiwer.wer(refs, hyps)
print(f'=== Kraken rec_v2 (default seg): CER {cer*100:.2f}% | WER {wer*100:.2f}% ===')
print('Baseline Tesseract: CER 33.11% | WER 81.63%')
result = {'system': 'kraken_rec_v2 (polish_nfd_9313 + ocr-pl-lines + ehri-pl-lines + v1 regions, default blla seg)',
          'split': 'impact-print-v2-test', 'cer_micro': cer, 'wer_micro': wer,
          'pages': len(records), 'best_model': best_model}
(bench / 'kraken_rec_v2_result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print('Zapisano:', bench / 'kraken_rec_v2_result.json')

In [ ]:
# 7. Upload wyniku i modelu na HF (HF_TOKEN z sekretu lub notebook_login)
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault('HF_TOKEN', UserSecretsClient().get_secret('HF_TOKEN'))
except Exception:
    pass
if not os.environ.get('HF_TOKEN'):
    from huggingface_hub import notebook_login
    notebook_login()
from huggingface_hub import upload_file
bench = WORKDIR / 'impact-print-v2'
upload_file(path_or_fileobj=str(bench / 'kraken_rec_v2_result.json'),
            path_in_repo='results/kraken_rec_v2.json',
            repo_id='PiotrSty/impact-print-v2', repo_type='dataset')
upload_file(path_or_fileobj=best_model,
            path_in_repo='models/kraken_rec_v2.safetensors',
            repo_id='PiotrSty/impact-print-v2', repo_type='dataset')
print('Uploaded: results/kraken_rec_v2.json + models/kraken_rec_v2.safetensors')